# Milestone 1 

**Course:** Introduction to DL and GenAI Project
**Competition:** Smart MCQ Solver Challenge
**Roll No:** 22f3000114

---

## Milestone 1 Objectives (per course spec)
1. **Text cleaning, tokenization, and handling missing data**
2. **Generate embeddings** using baseline models: **TF-IDF** and **Word2Vec**
3. **Compute cosine similarity** between the `prompt` and each `option`
4. **Calculate MAP@3** and understand the concept of mAP

## Design idea
Each question has a `prompt` and five options `A`–`E`; exactly one is correct.
Our Milestone-1 hypothesis is the natural NLP baseline:
> *the correct option is the one most **semantically similar** to the prompt.*

We embed the prompt and each option (first with TF-IDF, then with Word2Vec trained
from scratch on our corpus), rank the five options by cosine similarity to the prompt,
take the top-3, and evaluate with MAP@3. These non-neural, non-pretrained baselines
establish the performance **floor** that the later milestones (transformers, RAG,
fine-tuning) must beat.

## Tech stack
`pandas`, `scikit-learn` (TF-IDF, cosine similarity), `gensim` (Word2Vec — the
standard library for the Word2Vec algorithm named in the milestone spec), and
`wandb` for experiment tracking as required by the project.


## 0. Environment Setup

In [1]:
# gensim provides the Word2Vec implementation named in the milestone spec.
# wandb is required by the project for experiment tracking.
!pip install gensim wandb -qq

## 1. Weights & Biases Setup

We authenticate using the Kaggle Secrets store (same pattern used across the course
notebooks) so the API key is never hard-coded. The W&B key is stored under the secret
label `WANDB`, and runs are logged to the project team.

In [2]:
from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB")
wandb.login(key=wandb_key)

WANDB_ENTITY  = "22f3000114-dl-genai-project"
WANDB_PROJECT = "dl-genai-project"
print("W&B authenticated.")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


W&B authenticated.


## 2. Imports & Data Loading

In [3]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import glob

# Robust path finding so this runs regardless of how the competition data is nested
train_path = glob.glob('/kaggle/input/**/train.csv', recursive=True)[0]
print("Loading:", train_path)

df = pd.read_csv(train_path)
print("Shape:", df.shape)
df.head(3)

Loading: /kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
Shape: (2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


## 3. Handling Missing Data

A missing prompt or option would break vectorization, so we inspect first and fill any
gaps with an empty string. We also confirm all answer labels are valid A–E.

In [4]:
print("Missing values per column:")
print(df.isnull().sum())

text_cols = ['prompt', 'A', 'B', 'C', 'D', 'E']
df[text_cols] = df[text_cols].fillna('')

print("\nAnswer label distribution:")
print(df['answer'].value_counts())

Missing values per column:
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

Answer label distribution:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64


## 4. Text Cleaning & Tokenization

The prompts contain boilerplate framing such as *"Pick the best possible answer: ...
among the listed options."* We lowercase, remove punctuation, collapse whitespace, and
tokenize on word boundaries.

In [6]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)   # keep alphanumerics
    text = re.sub(r"\s+", " ", text).strip()    # collapse whitespace
    return text

def tokenize(text):
    return clean_text(text).split()

for col in text_cols:
    df[col + '_clean'] = df[col].apply(clean_text)

example = df['prompt'].iloc[0]
print("RAW    :", example[:90], "...")
print("CLEAN  :", clean_text(example)[:90], "...")
print("TOKENS :", tokenize(example)[:12])

RAW    : Pick the best possible answer: What is Martin Heidegger's view on the relationship between ...
CLEAN  : pick the best possible answer what is martin heidegger s view on the relationship between  ...
TOKENS : ['pick', 'the', 'best', 'possible', 'answer', 'what', 'is', 'martin', 'heidegger', 's', 'view', 'on']


## 5. MAP@3

**Mean Average Precision at 3.** With exactly one correct answer per question, Average
Precision at 3 reduces to a reciprocal-rank score:

| Rank of correct answer | Score |
|------------------------|-------|
| 1st | 1/1 = 1.000 |
| 2nd | 1/2 = 0.500 |
| 3rd | 1/3 = 0.333 |
| not in top-3 | 0.000 |

MAP@3 averages this across all questions. It rewards ranking the correct answer
**early**, which is precisely why we produce a *ranked* top-3 rather than a single guess.

In [7]:
def average_precision_at_3(predicted_labels, true_label):
    """AP@3 for one question with a single correct answer."""
    for rank, label in enumerate(predicted_labels[:3], start=1):
        if label == true_label:
            return 1.0 / rank
    return 0.0

def map_at_3(list_of_predictions, true_labels):
    """Mean AP@3 across all questions."""
    return np.mean([average_precision_at_3(p, t)
                    for p, t in zip(list_of_predictions, true_labels)])

# Verify implementation against the metric definition
assert average_precision_at_3(['A','B','C'], 'A') == 1.0
assert average_precision_at_3(['B','A','C'], 'A') == 0.5
assert average_precision_at_3(['C','D','A'], 'A') == 1/3
assert average_precision_at_3(['B','C','D'], 'A') == 0.0
print("MAP@3 implementation verified.")

MAP@3 implementation verified.


## 6. Baseline 0 

Before any NLP, the simplest possible predictor: always output the three most frequent
answer labels, in order of frequency. This is the true floor any real model must beat,
and it exploits any class imbalance in the label distribution.

In [8]:
freq_order = df['answer'].value_counts().index.tolist()
majority_pred = freq_order[:3]
print("Most frequent labels (ranked):", majority_pred)

majority_predictions = [majority_pred] * len(df)
majority_score = map_at_3(majority_predictions, df['answer'].tolist())
print(f"Majority-class MAP@3: {majority_score:.4f}")

Most frequent labels (ranked): ['B', 'C', 'A']
Majority-class MAP@3: 0.4213


## 7. Method 1: TF-IDF + Cosine Similarity

**TF-IDF** represents each text as a sparse vector weighting each term by how frequent
it is in that text but how rare it is across the corpus. Words shared by many documents
get down-weighted; distinctive words get up-weighted.

We fit one TF-IDF vocabulary on **all** prompts + options together so every text lives
in the same vector space, then rank each question's options by cosine similarity to its
prompt.

In [9]:
all_texts = []
for col in text_cols:
    all_texts.extend(df[col + '_clean'].tolist())

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000)
tfidf.fit(all_texts)
print(f"TF-IDF vocabulary size: {len(tfidf.vocabulary_)}")

def predict_tfidf(row):
    prompt_vec  = tfidf.transform([row['prompt_clean']])
    option_vecs = tfidf.transform([row[c + '_clean'] for c in ['A','B','C','D','E']])
    sims = cosine_similarity(prompt_vec, option_vecs)[0]     # 5 similarity scores
    ranked_idx = np.argsort(sims)[::-1]                      # high -> low
    labels = ['A','B','C','D','E']
    return [labels[i] for i in ranked_idx[:3]]

tfidf_predictions = df.apply(predict_tfidf, axis=1).tolist()
tfidf_score = map_at_3(tfidf_predictions, df['answer'].tolist())
print(f"TF-IDF cosine MAP@3: {tfidf_score:.4f}")

TF-IDF vocabulary size: 12464
TF-IDF cosine MAP@3: 0.2910


## 8. Method 2: Word2Vec + Cosine Similarity

**Word2Vec** learns dense vectors where words appearing in similar contexts get similar
vectors. Unlike TF-IDF (which only rewards **exact** shared words), Word2Vec can capture
looser semantic relatedness even when prompt and option use different vocabulary.

We train Word2Vec **from scratch** on our own corpus, no pretrained vectors, then
represent each text as the mean of its word vectors and rank options by cosine
similarity to the prompt.

In [10]:
from gensim.models import Word2Vec

tokenized_corpus = []
for col in text_cols:
    tokenized_corpus.extend([tokenize(t) for t in df[col].tolist()])

w2v = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=42,
    epochs=20
)
print(f"Word2Vec vocabulary size: {len(w2v.wv)}")

def embed_text(text):
    tokens = [t for t in tokenize(text) if t in w2v.wv]
    if not tokens:
        return np.zeros(w2v.vector_size)
    return np.mean([w2v.wv[t] for t in tokens], axis=0)

def predict_w2v(row):
    prompt_vec  = embed_text(row['prompt']).reshape(1, -1)
    option_vecs = np.vstack([embed_text(row[c]) for c in ['A','B','C','D','E']])
    sims = cosine_similarity(prompt_vec, option_vecs)[0]
    ranked_idx = np.argsort(sims)[::-1]
    labels = ['A','B','C','D','E']
    return [labels[i] for i in ranked_idx[:3]]

w2v_predictions = df.apply(predict_w2v, axis=1).tolist()
w2v_score = map_at_3(w2v_predictions, df['answer'].tolist())
print(f"Word2Vec cosine MAP@3: {w2v_score:.4f}")

Word2Vec vocabulary size: 2970
Word2Vec cosine MAP@3: 0.3154


## 9. Log Results to Weights & Biases

We log each baseline as a W&B run under the project team so Milestone 1 contributes to
the required multi-run comparison for the L1 viva. Each run records its method and MAP@3.

In [11]:
results = {
    "majority_class":    majority_score,
    "tfidf_cosine":      tfidf_score,
    "word2vec_cosine":   w2v_score,
}

for method_name, score in results.items():
    run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        name=f"m1-{method_name}",
        group="milestone-1-baselines",
        job_type="baseline",
        config={
            "milestone": 1,
            "method": method_name,
            "metric": "MAP@3",
            "embedding": method_name.split("_")[0],
            "trainable_params": 0,          # non-neural baselines
            "pretrained": False,
        },
        reinit=True,
    )
    run.log({"map_at_3": score})
    run.summary["map_at_3"] = score
    run.finish()

print("Logged all Milestone 1 baselines to W&B.")

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


map_at_3,▁
map_at_3,0.42125


map_at_3,▁
map_at_3,0.291


map_at_3,▁
map_at_3,0.31542


Logged all Milestone 1 baselines to W&B.


## 10. Results Comparison

In [12]:
summary = pd.DataFrame({
    'Method': ['Majority class', 'TF-IDF + cosine', 'Word2Vec + cosine'],
    'MAP@3':  [majority_score,   tfidf_score,       w2v_score]
}).sort_values('MAP@3', ascending=False).reset_index(drop=True)

print(summary.to_string(index=False))

           Method    MAP@3
   Majority class 0.421250
Word2Vec + cosine 0.315417
  TF-IDF + cosine 0.291000


## 11. Observations & Analysis

**Key finding: similarity baselines underperform the majority-class floor.**
Both TF-IDF and Word2Vec similarity scored **below** the majority-class baseline.

- The questions are **closed-book factual recall** (physics, philosophy). The correct
  option is *not* reliably the one sharing the most words / nearest embedding to the
  prompt.
- Distractors are **deliberately engineered** to be lexically and semantically close to
  the prompt, so "most similar to prompt" is a weak signal.
- **TF-IDF vs Word2Vec:** Word2Vec (0.3154) marginally beat TF-IDF (0.2910), likely because TF-IDF only rewards exact word overlap whereas Word2Vec's averaged embeddings capture softer semantic relatedness and are less fooled by surface matching. The gap is small and both remain well below the majority-class floor (0.4213), since neither can supply the factual knowledge these questions require; Word2Vec's quality is also limited by our tiny ~2000-row corpus, which Milestone 2's pretrained embeddings will address.

## Next Steps
- **Milestone 2:** replace these baselines with pretrained transformer embeddings and
  zero-shot classification; log runs to the same W&B project for comparison.
